In [ ]:
import os
# @title # Установка
#@markdown ---
# @markdown * Подключить гугл диск
mount_drive = False # @param {"type":"boolean"}
repo_url = "https://github.com/noblebarkrr/mvsepless"
home_dir = os.path.join(os.sep, "content")
mvsepless_dir = os.path.join(home_dir, "mvsepless-epsilon")
%cd $home_dir
!git clone $repo_url -b epsilon $mvsepless_dir
%cd $mvsepless_dir
!pip install --no-cache-dir uv
req = """
torch
torchvision
torchaudio
numpy==2.0.2
pandas
scipy
librosa
samplerate==0.1.0
matplotlib
tqdm
einops
protobuf
soundfile
pydub
webrtcvad
audiomentations
pedalboard==0.8.2
ml_collections
timm
wandb
accelerate
bitsandbytes
tokenizers
huggingface-hub
transformers
torchseg
demucs==4.0.0
asteroid
prodigyopt
torch_log_wmse
rotary_embedding_torch
gradio<6.0
omegaconf
beartype
spafe
torch_audiomentations
auraloss
onnx>=1.17
onnx2torch>=0.3.0
onnxruntime-gpu>=1.17
ml_dtypes
resampy
yt_dlp
pyngrok
tabulate
neuraloperator==1.0.2
torchcrepe
praat-parselmouth
faiss-cpu==1.11
local-attention
tenacity
pyworld
gdown
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
  f.write(req)
!uv pip install --no-cache-dir -qq -r requirements.txt
%cd $mvsepless_dir
if mount_drive:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
import os
from pyngrok import ngrok
import random
import string
import re
import urllib
import time
import ipywidgets as widgets
from IPython.display import display, Javascript
import threading
import subprocess

%cd $mvsepless_dir
#@title # Web-UI
#@markdown ---
port = 7862
#@markdown * Способ поделится приложением
sharing_method = "gradio" # @param ["gradio","ngrok","localtunnel","not"]
#@markdown * Токен для ngrok *(где взять его - https://dashboard.ngrok.com/get-started/your-authtoken)*
ngrok_token = "" # @param {"type":"string"}
#@markdown * Включить Vbach в Web-UI
vbach = False # @param {"type":"boolean"}

lt_sub_domain = "mvsepless"
def generate_subdomain(length=8):
    """Генерация случайного субдомена заданной длины"""
    chars = string.ascii_lowercase + string.digits
    return ''.join(random.choice(chars) for _ in range(length))

if sharing_method == "ngrok":
    try:
        ngrok.set_auth_token(ngrok_token)
        ngrok.kill()
        tunnel = ngrok.connect(port)
        print(f"Публичная ссылка: {tunnel.public_url}")
    except KeyboardInterrupt:
        ngrok.kill()

if sharing_method == "localtunnel":
    os.system("npm install -g localtunnel &>/dev/null")
    time.sleep(7)
    with open('url.txt', 'w') as file:
        file.write('')
    subdomain = f"{re.sub(r'[^a-zA-Z0-9]', '', lt_sub_domain)}-{generate_subdomain(25)}"

    # Флаг для контроля работы потока
    tunnel_running = True

    def run_tunnel():
        while tunnel_running:
            print("localtunnel включается...")
            try:
                # Используем subprocess вместо os.system для лучшего контроля
                process = subprocess.Popen(
                    f'lt --port {port} '
                    f'{f"--subdomain {subdomain}" if lt_sub_domain != "" and not lt_sub_domain.isspace() else ""}',
                    shell=True,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE
                )
                process.wait()  # Ждем завершения процесса
                if not tunnel_running:
                    break
                time.sleep(5)  # Пауза перед перезапуском
            except Exception as e:
                if tunnel_running:
                    print(f"Ошибка в localtunnel: {e}")
                    time.sleep(5)

    tunnel_thread = threading.Thread(target=run_tunnel, daemon=True)
    tunnel_thread.start()

    time.sleep(3)
    try:
        endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        tunnel_url = f"https://{subdomain}.loca.lt"
        print(f"Публичная ссылка: {tunnel_url}")

        # Создаем текстовое поле с URL, а не IP
        text_field = widgets.Text(
            value=endpoint_ip,  # Исправлено: показываем URL, а не IP
            description='URL:',
            disabled=True
        )
        text_field.add_class("copy-enabled")

        display(text_field)

        # Исправленный JavaScript для копирования
        display(Javascript("""
        setTimeout(() => {
            const input = document.querySelector('.copy-enabled input');
            if (!input) return;

            const btn = document.createElement('button');
            btn.innerHTML = '📋';
            btn.style.cssText = `
                margin-left: 8px;
                border: none;
                background: none;
                cursor: pointer;
                font-size: 1.2em;
            `;
            input.parentNode.appendChild(btn);

            btn.addEventListener('click', () => {
                navigator.clipboard.writeText(input.value)  // Исправлено: input.value вместо input
                    .then(() => {
                        btn.innerHTML = '✓';
                        setTimeout(() => btn.innerHTML = '📋', 2000);
                    })
                    .catch(err => {
                        console.error('Ошибка копирования: ', err);
                    });
            });
        }, 300);
        """))

    except Exception as e:
        print(f"Ошибка при старте localtunnel: {e}")

    # Функция для корректного завершения
    def stop_tunnel():
        global tunnel_running
        tunnel_running = False
        print("Localtunnel завершает работу...")

    # Регистрируем обработчик для Ctrl+C
    import signal
    original_signal_handler = signal.getsignal(signal.SIGINT)

    def signal_handler(sig, frame):
        stop_tunnel()
        # Восстанавливаем оригинальный обработчик и вызываем его
        signal.signal(signal.SIGINT, original_signal_handler)
        raise KeyboardInterrupt

    signal.signal(signal.SIGINT, signal_handler)

share_arg = "--share" if sharing_method == "gradio" else ""
vbach_arg = "--vbach" if vbach else ""
!python mvsepless/app.py --port $port $share_arg --add_app $vbach_arg --use_plugins

# MVSepLess CLI

In [ ]:
#@markdown ---
#@markdown ### Входные данные
#@markdown * Путь к входной папке/файлу:
input_path = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
#@markdown ---
#@markdown ### Выбор модели
#@markdown * Тип / Имя модели:
model_name = "bs_6stem" # @param ['mbr_vocals_kim', 'mbr_wsa', 'mbr_instvoc_duality1_unwa', 'mbr_instvoc_duality2_unwa', 'mbr_kimft1_unwa', 'mbr_kimft2_unwa', 'mbr_kimft2b_unwa', 'mbr_kimft3_prev_unwa', 'mbr_bigbeta1_unwa', 'mbr_bigbeta2_unwa', 'mbr_bigbeta3_unwa', 'mbr_bigbeta4_unwa', 'mbr_bigbeta5e_unwa', 'mbr_bigbeta6_unwa', 'mbr_bigbeta6x_unwa', 'mbr_inst1_unwa', 'mbr_inst1+_unwa', 'mbr_inst1e_unwa', 'mbr_inst1e+_unwa', 'mbr_inst2_unwa', 'mbr_small_unwa', 'mbr_bleed_supressor_unwa_97chris', 'mbr_inst_becruily', 'mbr_guitar_becruily', 'mbr_karaoke_becruily', 'mbr_vocals_becruily', 'mbr_deux_becruily', 'mbr_syhft1', 'mbr_syhft2', 'mbr_syhft2.5', 'mbr_syhft3', 'mbr_bigsyhft1fast', 'mbr_syhftbeta1', 'mbr_syhftB1_1', 'mbr_syhftB1_2', 'mbr_syhftB1_3', 'mbr_syhft_4stem', 'mbr_syhft_4stem2', 'mbr_inst_1652_essid', 'mbr_inst_1681_essid', 'mbr_instfv1_gabox', 'mbr_instfv2_gabox', 'mbr_instfv3_gabox', 'mbr_instfv4_gabox', 'mbr_instfv4n_gabox', 'mbr_instfv5_gabox', 'mbr_instfv5n_gabox', 'mbr_instfv6_gabox', 'mbr_instfv6n_gabox', 'mbr_instfv7_gabox', 'mbr_instfv7n_gabox', 'mbr_instfv7+_gabox', 'mbr_instfv7z_gabox', 'mbr_instfv8_gabox', 'mbr_instfv8b_gabox', 'mbr_instfv9_gabox', 'mbr_instfv9_2_gabox', 'mbr_instfv10_gabox', 'mbr_instflowersv10_gabox', 'mbr_instfvx_gabox', 'mbr_instbv1_gabox', 'mbr_instbv2_gabox', 'mbr_instbv3_gabox', 'mbr_vocalsfv1_gabox', 'mbr_vocalsfv2_gabox', 'mbr_vocalsfv3_gabox', 'mbr_vocalsfv4_gabox', 'mbr_vocalsfv5_gabox', 'mbr_vocalsfv6_gabox', 'mbr_vocalsfv7_beta1_gabox', 'mbr_vocalsfv7_beta2_gabox', 'mbr_vocalsfv7_beta3_gabox', 'mbr_karaoke25022025_gabox', 'mbr_karaoke28022025_gabox', 'mbr_karaoke1_gabox', 'mbr_karaoke2_gabox', 'mbr_leadvoc_dereverb_gabox', 'mbr_denoise_debleed_gabox', 'mbr_karaoke_fusion_gonzaluigi', 'mbr_karaoke_fusion_aggr_gonzaluigi', 'mbr_bve_gonzaluigi', 'mbr_karaoke_fusion2_aggr_gonzaluigi', 'mbr_karaoke_fusion_total_aggr_gonzaluigi', 'mbr_dereverb_anvuew', 'mbr_dereverb_less_aggr_anvuew', 'mbr_dereverb_mono_anvuew', 'mbr_aspiration_sucial', 'mbr_dereverb_echo1_sucial', 'mbr_debigreverb_sucial', 'mbr_desuperbigreverb_sucial', 'mbr_dereverb-echo_fused_sucial', 'mbr_dereverb-echo2_sucial', 'mbr_karaoke_aufr33_viperx', 'mbr_denoise_aufr33', 'mbr_denoise_aggr_aufr33', 'mbr_crowd_aufr33_viperx', 'mbr_vocals_viperx', 'mbr_vocalsf_aname', 'mbr_kimft1_aname', 'mbr_kimft2_aname', 'mbr_kimft2f_aname', 'mbr_kimft3_aname', 'mbr_small_aname', 'mbr_duality1_aname', 'mbr_4stemlarge1_aname', 'mbr_4stemlarge2_aname', 'mbr_4stemxl1_aname', 'mbr_percussion_yolkispaliks', 'mbr_inst_metal_prev_meskvlla33', 'mbr_neo_inst_vfx', 'bs_cr_4stem_zf_turbo', 'bs_drums_beatloo_labs', 'bs_bass_beatloo_labs', 'bs_vocals_1296_viperx', 'bs_other_viperx', 'bs_revive1_unwa', 'bs_revive2_unwa', 'bs_revive3e_unwa', 'bs_resurrection_unwa', 'bs_resurrection_inst_unwa', 'bs_inst_fno_unwa', 'bs_inst_hyperace_unwa', 'bs_inst_hyperace2_unwa', 'bs_voc_hyperace2_unwa', 'bs_karaoke_becruily', 'bs_voctest_gabox', 'bs_karaoke_gabox', 'bs_6stem', 'bs_6stem_fixed', 'bs_logic_6stem', 'bs_4stem_zfturbo', 'bs_4stemft_syh99999', 'bs_male_female_146_sucial', 'bs_male_female_267_sucial', 'bs_male_female_aufr33', 'bs_deverb_256_8_anvuew', 'bs_deverb_384_10_anvuew', 'bs_deverb_room_anvuew', 'bs_karaoke_anvuew', 'bs_vocals_anvuew', 'bs_4stem_aname', 'mdx23c_instvoc_zfturbo', 'mdx23c_instvoc_hq1', 'mdx23c_instvoc_hq2', 'mdx23c_d1581', 'mdx23c_drumsep_6stem_aufr33_jarredou', 'mdx23c_drumsep_5stem_aufr33_jarredou', 'mdx23c_dereverb_aufr33_jarredou', 'mdx23c_mid_side_wesleyr36', 'mdx23c_4stem_zfturbo', 'mdx23c_orch_verosment', 'mdx_kim_inst', 'mdx_kim_vocal1', 'mdx_kim_vocal2', 'mdx_kuielab_a_bass', 'mdx_kuielab_a_drums', 'mdx_kuielab_a_other', 'mdx_kuielab_a_vocals', 'mdx_kuielab_b_bass', 'mdx_kuielab_b_drums', 'mdx_kuielab_b_other', 'mdx_kuielab_b_vocals', 'mdx_reverb_hq_foxjoy', 'mdx_inst1', 'mdx_inst2', 'mdx_inst3', 'mdx_inst_full_292', 'mdx_inst_hq1', 'mdx_inst_hq2', 'mdx_inst_hq3', 'mdx_inst_hq4', 'mdx_inst_hq5', 'mdx_inst_main', 'mdx_vocft', 'mdx_crowd_hq1', 'mdx_inst_187_beta', 'mdx_inst_82_beta', 'mdx_inst_90_beta', 'mdx_main_340', 'mdx_main_390', 'mdx_main_406', 'mdx_main_427', 'mdx_main_438', 'mdx_1_9703', 'mdx_2_9682', 'mdx_3_9662', 'mdx_9482', 'mdx_karaoke1', 'mdx_karaoke2', 'mdx_main', '1_hp-uvr', '2_hp-uvr', '3_hp-vocal-uvr', '4_hp-vocal-uvr', '5_hp-karaoke-uvr', '6_hp-karaoke-uvr', '7_hp2-uvr', '8_hp2-uvr', '9_hp2-uvr', '10_sp-uvr-2b-32000-1', '11_sp-uvr-2b-32000-2', '12_sp-uvr-3b-44100', '13_sp-uvr-4b-44100-1', '14_sp-uvr-4b-44100-2', '15_sp-uvr-mid-44100-1', '16_sp-uvr-mid-44100-2', '17_hp-wind_inst-uvr', 'uvr-de-echo-aggressive', 'uvr-de-echo-normal', 'uvr-deecho-dereverb', 'uvr-denoise-lite', 'uvr-denoise', 'uvr-bve-4b_sn-44100-1', 'uvr-bve-v2-4b-sn-44100', 'mgm-v5-karokee-32000-beta1', 'mgm-v5-karokee-32000-beta2-agr', 'mgm_highend_v4', 'mgm_lowend_a_v4', 'mgm_lowend_b_v4', 'mgm_main_v4', 'uvr-de-reverb-aufr33-jarredou', 'uvr-de-breath-sucial-v1', 'uvr-de-breath-sucial-v2', 'vr_harmonic_noise_sep', 'scnet_4stem_zfturbo', 'scnet_xl_ihf_4stem_zfturbo', 'scnet_xl_4stem_starrytong', 'scnet_xl_4stem_zftrubo', 'scnet_huge_4stem_aname', 'scnet_masked_small_4stem_zftrubo', 'scnet_masked_xl_ihf_4stem_zftrubo', 'scnet_tran_4stem_zftrubo', 'scnet_jazz_4stem_jorisvaneyghen', 'scnet_xl_jazz_4stem_jorisvaneyghen', 'scnet_choirsep_exp', 'demucs4_mvsep_vocals', 'demucs4_4stem', 'demucs4_6stem', 'demucs3_mmi', 'demucs4_ft_bass', 'demucs4_ft_drums', 'demucs4_ft_vocals', 'demucs4_ft_other', 'demucs_mid_side_wesleyr36', 'demucs4_choirsep', 'bandit_plus', 'bandit_v2_multi']

# @markdown ---
# @markdown ### Настройки разделения
# @markdown * Извлечь  инструментал:
instrumental = True # @param {type:"boolean"}
#@markdown ---
#@markdown ### Выходные данные
#@markdown * Формат:
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
# @markdown * Битрейт
bitrate = 320 # @param {"type":"slider","min":32,"max":320,"step":1}
# @markdown * Выбрать выходные стемы(через пробел, например ("vocal"  "instrumental")):
stems_to_extract = "" # @param {type:"string"}
# @markdown * Шаблон именования выходных файлов:
output_template = "NAME (STEM) MODEL" # @param {type:"string"}
#@markdown * Путь к выходной папке:
output_dir = "/content/output" # @param {"type":"string","placeholder":"/путь/к/папке"}

%cd $mvsepless_dir

cmd = [
    "python",
    "mvsepless/separator.py",
    f"--input \"{input_path}\"",
    f"--output_dir \"{output_dir}\"",
    f"--model_name \"{model_name}\"",
    f"--output_format \"{output_format}\"",
    f"--output_bitrate \"{bitrate}k\"",
    f"--template \"{output_template}\""
]

if instrumental:
    cmd.append("--ext_inst")

if stems_to_extract:
    cmd.append(f"--selected_stems {stems_to_extract}")

!{" ".join(cmd)}

# VBach CLI

In [ ]:
#@title Показать список установленных моделей для преобразования
%cd $mvsepless_dir
!python mvsepless/vbach.py model_manager list

In [ ]:
#@title Удаление голосовой модели
%cd $mvsepless_dir
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
!python mvsepless/vbach.py model_manager remove --model_name "$voicemodel_name"

## Установка голосовой модели

In [ ]:
#@title Через локальные файлы
%cd $mvsepless_dir
pth_path = "" # @param {"type":"string","placeholder":"Путь к *.pth файлу"}
index_path = "" # @param {"type":"string","placeholder":"Путь к *.index файлу"}
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
index = f"--index \"{index_path}\"" if index_path != "" else ""
if pth_path != "" and voicemodel_name != "":
    !python mvsepless/vbach.py model_manager install_local --model_name "$voicemodel_name" --pth "$pth_path" $index

In [ ]:
#@title Через файлы с интернета
%cd $mvsepless_dir
pth_url = "" # @param {"type":"string","placeholder":"Ссылка на *.pth файл"}
index_url = "" # @param {"type":"string","placeholder":"Ссылка на *.index файл"}
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
index = f"--index_url \"{index_url}\"" if index_url != "" else ""
if pth_url != "" and voicemodel_name != "":
    !python mvsepless/vbach.py model_manager install_url_files --model_name "$voicemodel_name" --pth_url "$pth_url" $index

In [ ]:
#@title Через zip файл с интернета
%cd $mvsepless_dir
zip_url = "" # @param {"type":"string","placeholder":"Ссылка на zip файл"}
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
if zip_url != "" and voicemodel_name != "":
    !python mvsepless/vbach.py model_manager install_url_zip --model_name "$voicemodel_name" --url "$zip_url"

## Инференс

In [ ]:
#@markdown ---
#@markdown ### Входные данные
#@markdown * Путь к входной папке/файлу:
input_path = "" # @param {"type":"string","placeholder":"/путь/к/файлу"}
#@markdown * Имя модели:
voicemodel_name = "" # @param {"type":"string","placeholder":"Имя модели"}
# @markdown ---
# @markdown  ### Hubert
# @markdown * Стэк
stack = "fairseq" # @param ["fairseq","transformers"]
# @markdown * Имя модели для fairseq
fairseq_embedder = "hubert_base" # @param ["hubert_base","contentvec_base","korean_hubert_base","chinese_hubert_base","portuguese_hubert_base","japanese_hubert_base"]
# @markdown * Имя модели для transformers
transformers_embedder = "contentvec" # @param ["contentvec","spin","spin-v2","chinese-hubert-base","japanese-hubert-base","korean-hubert-base"]
# @markdown ---
# @markdown  ### Настройки преобразования
# @markdown * Влияние индекса
index_rate = 1 # @param {"type":"slider","min":0,"max":1,"step":0.01}
# @markdown * Стерео режим
stereo_mode = "mono" # @param ["mono","left/right","sim/dif"]
# @markdown * Метод определения тона
method_pitch = "rmvpe+" # @param ["rmvpe+","mangio-crepe","mangio-crepe-tiny","fcpe",'harvest","pm","pyin"]
# @markdown * Изменение высоты тона (полутона)
pitch = 0 # @param {"type":"slider","min":-48,"max":48,"step":1}
# @markdown * Длина шага (для mangio-crepe)
hop_length = 128 # @param {"type":"slider","min":8,"max":512,"step":8}
# @markdown * Радиус фильтра
filter_radius = 3 # @param {"type":"slider","min":1,"max":7,"step":1}
# @markdown * Соотношение огибающих громкости
rms = 0.25 # @param {"type":"slider","min":0,"max":1,"step":0.01}
# @markdown * Защита согласных
protect = 0.33 # @param {"type":"slider","min":0,"max":0.5,"step":0.01}
# @markdown ---
#@markdown ### Дополнительные настройки
# @markdown * Минимальная частота F0
f0_min = 50 # @param {type:"integer"}
# @markdown * Максимальная частота F0
f0_max = 1100 # @param {type:"integer"}
# @markdown ---
#@markdown ### Выходные данные
#@markdown * Формат:
output_format = "mp3" # @param ["mp3", "wav", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
# @markdown * Имя выходного файла:
output_name = "F0METHOD_PITCH_(MODEL)_NAME" # @param {type:"string"}
#@markdown * Путь к выходной папке:
output_dir = "/content/vbach_output" # @param {"type":"string","placeholder":"/путь/к/папке"}



%cd $mvsepless_dir

cmd = [
    "python",
    "mvsepless/vbach.py", "cli",
    f"--input \"{input_path}\"",
    f"--output_dir \"{output_dir}\"",
    f"--model_name \"{voicemodel_name}\"",
    f"--output_format \"{output_format}\"",
    f"--index_rate {index_rate}",
    f"--output_name \"{output_name}\"",
    "--format_name",
    f"--stereo_mode {stereo_mode}",
    f"--method_pitch {method_pitch}",
    f"--pitch {pitch}",
    f"--hop_length {hop_length}",
    f"--filter_radius {filter_radius}",
    f"--rms {rms}",
    f"--protect {protect}",
    f"--f0_min {f0_min}",
    f"--f0_max {f0_max}",
    f"--embedder_name {fairseq_embedder}" if stack == "fairseq" else f"--embedder_name {transformers_embedder} --use_transformers"
]

!{" ".join(cmd)}